# Client and server: the server never holds the secret

The client generates the keys the server's plan needs, keeps the secret, encrypts each input and decrypts each
output; the server runs the encrypted model on the GPU from the client's key bundle alone (`docs/WIRE.md`). Here the
server is GPT-2's planned decode (`perseus.serve.gpt2_decode`) in a separate process, and the client runs in this
notebook without touching a GPU.

Prerequisites: the exported GPT-2 weights (`WEIGHTS_PATH`) and the reference outputs (`ALL_BLOCKS_IO_DIR`),
README step 1; the server process needs the GPU.

In [ ]:
import json, os, subprocess, sys, tempfile, time
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "bootstrap_placements").is_dir() else Path.cwd().parent
DATA = Path(os.environ.get("PERSEUS_DATA", REPO / ".cache"))
env = {**os.environ,
       "WEIGHTS_PATH": os.environ.get("WEIGHTS_PATH", str(DATA / "models/openai-community/gpt2/classic/weights.bin.zip")),
       "ALL_BLOCKS_IO_DIR": os.environ.get("ALL_BLOCKS_IO_DIR", str(DATA / "oracle/gpt2/all_blocks_io"))}
work = Path(tempfile.mkdtemp(prefix="perseus_served_"))
sock = str(work / "gpt2.sock")
TOKENS = int(os.environ.get("NB_TOKENS", "4"))

## The server

It compiles the plan's programs from a CPU twin, then waits for a client: the session opens on the client's key
bundle (no secret key on the server).

In [ ]:
server = subprocess.Popen([sys.executable, "-m", "perseus.serve.gpt2_decode", "server", "--addr", sock],
                          cwd=REPO, env=env, stdout=open(work / "server.log", "w"), stderr=subprocess.STDOUT)
print("server pid", server.pid, "log", work / "server.log")

## The client

No device: `CUDA_VISIBLE_DEVICES` is empty for the client process. It asks the server for its key manifest,
generates the bundle, then per token encrypts block 0's input, sends it, and decrypts the logits (and, with
`--argmax`, the encrypted argmax's one-hot) it gets back.

In [ ]:
out = work / "served.json"
client_env = {**env, "CUDA_VISIBLE_DEVICES": ""}
r = subprocess.run([sys.executable, "-m", "perseus.serve.gpt2_decode", "client", "--addr", sock,
                    "--tokens", str(TOKENS), "--argmax", "--out", str(out)],
                   cwd=REPO, env=client_env, capture_output=True, text=True)
print(r.stdout[-3000:])
server.wait(timeout=600)
res = json.loads(out.read_text())
print(json.dumps(res, indent=1)[:2000])